In [1]:
import pandas as pd
import sqlite3

Parsing SQLite tables to cleaned dataframes

In [2]:
dbconn = sqlite3.connect('location.db')
tables = list(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", dbconn)['name'])

In [3]:
query_1 = """
SELECT DISTINCT MAX(F.timestamp) AS max_timestamp, F.stopId, G.tripId
FROM (
    SELECT *
    FROM (
        SELECT MAX(timestamp) AS max_timestamp, stopId, [trip.tripId] AS tripId FROM 
"""

query_2 = """
        WHERE [trip.routeId] = 6641 AND [trip.directionId] = 1 AND (stopId = 11606 OR stopId = 12483)
        GROUP BY tripId, stopId
        )
    GROUP BY tripId
    HAVING COUNT(*) = 2
    ) G
INNER JOIN 
"""
query_3 = """
F ON F.[trip.tripId] = G.tripId
WHERE F.stopId = 11606 OR F.stopId = 12483
GROUP BY G.tripId, F.stopId
ORDER BY tripId
"""
location_dfs = []

for table in tables:
    table_name = "\'" + table + "\'"
    query = query_1 + table_name + query_2 + table_name + query_3
    
    location_dfs.append(pd.read_sql_query(query, dbconn))

In [4]:
location_dfs[0].head()

,max_timestamp,stopId,tripId
0,1776351675,11606,14898260
1,1776354503,12483,14898260
2,1776351851,11606,14898261
3,1776354509,12483,14898261
4,1776352088,11606,14898262


Cleaning & transforming timestamps into relevant delta times

In [5]:
delta_time_dfs = []

for df in location_dfs:
    delta_df = df
    
    delta_df['start_time'] = pd.to_datetime(delta_df['max_timestamp'], unit = 's', errors = 'coerce')
    delta_df['ending_time'] = delta_df.groupby('tripId')['start_time'].shift(-1)
    delta_df['delta_time'] = (delta_df['ending_time'] - delta_df['start_time'])
    delta_df = delta_df[pd.notnull(delta_df['delta_time'])]
    delta_df['delta_time_min'] = delta_df['delta_time'].dt.total_seconds().div(60)

    cleaned_df = delta_df[['tripId', 'delta_time_min']]
        
    delta_time_dfs.append(cleaned_df)

In [6]:
delta_time_dfs[0].head()

,tripId,delta_time_min
0,14898260,47.133333
2,14898261,44.300000
4,14898262,43.266667
6,14898263,40.666667
8,14898264,46.466667
